### 6. Text Clustering Projects

**Non-Sequential Data**
- if o/p is available : labelled Data --> supervised ML (regression or classification)
- if o/p is not available : unlabelled Data --> unsupervised ML (clustering)

In [2]:
import pandas as pd

In [3]:
npr = pd.read_csv('npr.csv')
npr.head()

,Article
0,"In the Washington of 2016, even when the polic..."
1,Donald Trump has used Twitter — his prefe...
2,Donald Trump is unabashedly praising Russian...
3,"Updated at 2:50 p. m. ET, Russian President Vl..."
4,"From photography, illustration and video, to d..."


In [4]:
npr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 1 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Article  200 non-null    object
dtypes: object(1)
memory usage: 1.7+ KB


#### Text Preprocessing
##### Text Cleaning
One of the main reason is to reduce the input columns
- Remove Punctuation
- Remove Stopwords
- Stemming

In [16]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
wnl = WordNetLemmatizer()

corpus = []
for i in range(0, len(npr)):
    rp = re.sub('[^a-zA-Z]', ' ', npr['Article'][i])
    rp = rp.lower()
    rp = rp.split()
    rp = [wnl.lemmatize(word) for word in rp if not word in set(stopwords.words('english'))]
    rp = ' '.join(rp)
    corpus.append(rp)

print(corpus[0])

washington even policy bipartisan politics cannot sense year show little sign ending dec president obama moved sanction russia alleged interference u election concluded republican long called similar severe measure could scarcely bring approve house speaker paul ryan called obama measure appropriate also overdue prime example administration ineffective foreign policy left america weaker eye world gop leader sounded much theme urging president obama year take strong action deter russia worldwide aggression including operation wrote rep devin nunes chairman house intelligence committee week left office president suddenly decided stronger measure indeed warranted appearing cnn frequent obama critic trent frank called much tougher action said three time obama finally found tongue meanwhile fox news various spokesman trump said obama real target russian man poised take white house less three week spoke obama trying tie trump hand box meaning would forced either keep sanction odds republican

##### Vectorization

In [6]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()
X = cv.fit_transform(corpus).toarray()
X

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(200, 12008))

In [7]:
X.shape

(200, 12008)

#### Modelling using LDA

In [8]:
from sklearn.decomposition import LatentDirichletAllocation

model = LatentDirichletAllocation(n_components=3)
model.fit(X)

# In this algorithm we have to specify the number of topics we want to find. 
# Here we are looking for 3 topics. This works by finding the most common word in 
# each topic and then grouping the articles based on those words.
# Euclidean Distance is used to find the distance between the articles and the topics.
# The closer the article is to the topic, the more likely it is to belong to that topic.

,n_components,3
,doc_topic_prior,None
,topic_word_prior,None
,learning_method,'batch'
,learning_decay,0.7
,learning_offset,10.0
,max_iter,10
,batch_size,128
,evaluate_every,-1
,total_samples,1000000.0
,perp_tol,0.1


In [9]:
topic_results = model.transform(X)
# Here we have 3 topics and the values represent the 
# probability of each article belonging to each topic

In [10]:
topic_results[0]

array([9.98839235e-01, 5.50194547e-04, 6.10570444e-04])

In [11]:
topic_results[0].argmax()
# This means that the first article belongs to the second topic (index 1) 
# because it has the highest probability.

np.int64(0)

##### Combinining with Original Data

In [12]:
npr['group'] = topic_results.argmax(axis=1)
npr.head()
# Here we are adding a new column to the original dataframe 
# that contains the topic group for each article.

,Article,group
0,"In the Washington of 2016, even when the polic...",0
1,Donald Trump has used Twitter — his prefe...,0
2,Donald Trump is unabashedly praising Russian...,0
3,"Updated at 2:50 p. m. ET, Russian President Vl...",0
4,"From photography, illustration and video, to d...",0


Showing top words per topic

In [13]:
for index, topic in enumerate(model.components_):
    print(f'The top 10 words for topic {index} are:')
    print([cv.get_feature_names_out()[i] for i in topic.argsort()[-10:]])
    print('\n')

The top 10 words for topic 0 are:
['would', 'time', 'new', 'said', 'people', 'like', 'one', 'year', 'trump', 'say']


The top 10 words for topic 1 are:
['would', 'way', 'new', 'time', 'like', 'said', 'year', 'one', 'people', 'say']


The top 10 words for topic 2 are:
['trump', 'new', 'one', 'people', 'republican', 'like', 'time', 'year', 'said', 'say']




In [15]:
npr[npr['group'] == 0]

,Article,group
0,"In the Washington of 2016, even when the polic...",0
1,Donald Trump has used Twitter — his prefe...,0
2,Donald Trump is unabashedly praising Russian...,0
3,"Updated at 2:50 p. m. ET, Russian President Vl...",0
4,"From photography, illustration and video, to d...",0
...,...,...
195,Mothers should feel comfortable infants in p...,0
196,"In South Korea, preparing for the worst has be...",0
197,David Bowie had long wanted to make a record w...,0
198,Chances are your doctor has stopped taking not...,0
